In [3]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

from matplotlib import pyplot as plt

# --- Zufällige Datenmatrix ---
np.random.seed(420)
X, y = make_blobs(n_samples=100, n_features=2, centers=3, random_state=42)

# --- Visualisierung ---
# plt.scatter(*X.T, c=y)
# plt.show()

# --- Zuweisung zufälliger Cluster ---
K = 3
y = np.random.randint(low=0, high=K, size=X.shape[0])

# --- Visualisierung ---
# plt.scatter(*X.T, c=y)
# plt.show()

# --- Clusterzentren ---
centroids = np.array([np.mean(X[y==c], axis=0) for c in np.unique(y)])

# --- Visualisierung ---
# plt.scatter(*X.T, c=y)
# plt.scatter(*centroids.T, s=100, c=range(centroids.shape[0]), edgecolor="black")
# plt.show()

# --- Zuweisung der Datenpunkt ---
y = np.array([np.argmin([(x - centroid) @ (x - centroid) for centroid in centroids]) for x in X])
centroids = np.array([np.mean(X[y==c], axis=0) for c in np.unique(y)])
# --- Wiederhole bis sich y nicht mehr verändert ---

# --- Vergleichswert ---
# kmeans = KMeans(n_clusters=K, n_init=20)
# y_ = kmeans.fit_predict(X)
# centroids_ = kmeans.cluster_centers_

# --- Visualisierung ---
# plt.scatter(*X.T, c=y)
# plt.scatter(*X.T, marker="x", c=y_)
# plt.scatter(*centroids.T, s=100, c=range(centroids.shape[0]), edgecolor="black", label="sklearn")
# plt.scatter(*centroids_.T, marker="x", s=100, c="black", label="sklearn")
# plt.legend()
# plt.show()

# Optional: Berechne icv
# --- Inter-Cluster-Variation -> icv ---
# icv = sum([1 / np.array([y==c], dtype=int).sum() * sum([sum([sum((x1 - x2)**2) for x2 in X[y==c]]) for x1 in X[y==c]]) for c in np.unique(y)])
# --- Wiederhole n-Mal, gebe kleinste icv zurück ---
# Optional: "Beste" Clusteranzahl, wenn die icv stark abgesunken ist

In [80]:
def kmeans(X, K=3):
    # --- Zuweisung zufälliger Cluster ---
    y = np.random.randint(low=0, high=K, size=X.shape[0])

    while True:
        # --- Clusterzentren ---
        centroids = np.array([np.mean(X[y==c], axis=0) for c in np.unique(y)])
        
        # --- Zuweisung der Datenpunkt ---
        y_pred = np.array([np.argmin([(x - centroid) @ (x - centroid) for centroid in centroids]) for x in X])

        # --- Wenn ein Cluster leer ist, verschiebe zufällige Punkte in das Cluster ---
        idx = np.array(range(y_pred.shape[0]))
        np.random.shuffle(idx)
        for c, i in zip(set(range(K)).difference(set(y_pred)), idx):
            y_pred[i] = c
            
        # --- Wiederhole bis sich y nicht mehr verändert ---
        if all(y == y_pred):
            return y_pred

        y = y_pred

In [81]:
# --- Inter-Cluster-Variation -> icv ---
def icv(X, y):
    return sum([1 / np.array([y==c], dtype=int).sum() * sum([sum([sum((x1 - x2)**2) for x2 in X[y==c]]) for x1 in X[y==c]]) for c in np.unique(y)])

In [82]:
# --- Wiederhole n-Mal, gebe kleinste icv zurück ---
# Obacht: Bringt hier nicht viel, da np.random.seed gesetzt ist
def kmeans_icv(X, K=3, n=10):
    y = kmeans(X, K)
    _icv = icv(X, y)

    for _ in range(1, n):
        _y = kmeans(X, K)
        if icv(X, _y) < _icv:
            y = _y
            _icv = icv(X, _y)
    
    return y

In [87]:
# --- icv nach Clusteranzahl ---
K = range(1, 9)
ICV = [icv(X, kmeans_icv(X, K=k)) for k in K]

# --- Visualisierung ---
# plt.plot(K, ICV)
# plt.xlabel("Clusteranzahl ($K$)")
# plt.ylabel("$icv_K$")
# plt.show()